# Palace results → pyEPR (this run)

This notebook was stamped into the run folder by the pipeline. Workflow:

1. When the Palace job on vanda finishes, copy the whole `postpro_<TAG>/`
   folder from the cluster back into **this folder** (the job's `.o` log
   too, for the run registry).
2. Run the cells top to bottom in the `qiskit_metal` env.

It feeds Palace's outputs — `eig.csv` (frequencies, Q) and `port-EPR.csv`
(the junction's signed energy-participation ratios, which Palace computes
because the JJ is an inductive `LumpedPort`) — into **pyEPR's real
`QuantumAnalysis.analyze_variation`** via `palace_epr.py` (stamped
alongside). Nothing is re-implemented: `chi_O1`, `chi_ND`, `f_ND`, `ZPF`,
`g` all come from the identical code path the Ansys `EPR_master_nb` uses.

Everything below auto-discovers from this folder: the postpro directory
(highest AMR iteration), the palace config, and the junction L/C from the
config's `LumpedPort` — so what enters pyEPR is exactly what the solver
used. You only ever edit `MODES`.

**Physics caveats:**

- On the **Order-1 mode-map track** frequencies wander a few hundred MHz
  between AMR passes; χ and α inherit that. Use those runs for mode
  identification and orders of magnitude; quote numbers from the Order-2
  track.
- The numerical diagonalization builds a `fock_trunc ** n_modes` Hilbert
  space. Keep `MODES` to ≲4 modes (qubit + the neighbours you care
  about), exactly as the Ansys notebook subsets via `modes_to_analyse`.
- If |p| is split across two nearly degenerate modes they are
  hybridised: compare sums over the pair, not per-mode values (see the
  `identify_modes` table).

In [ ]:
# ===================== auto-discovery (no pyEPR needed) ==============
import os, glob, json, re
import numpy as np
import pandas as pd

RUN_DIR = os.getcwd()

TAG   = None    # None = newest postpro_* here; or "run1"
MODES = None    # None = all modes if <=4, else first 4 (EDIT after
                # reading the identify_modes table below)
MODE_NAMES = None
COS_TRUNC, FOCK_TRUNC = 6, 7

# ---- postpro folder (and highest AMR iteration inside it) -----------
pp = sorted(d for d in glob.glob(os.path.join(RUN_DIR, "postpro_*"))
            if os.path.isdir(d))
if TAG is not None:
    pp = [d for d in pp if os.path.basename(d) == f"postpro_{TAG}"]
if not pp:
    raise FileNotFoundError(
        "no postpro_* folder here yet -- copy it back from vanda first")
if len(pp) > 1:
    print(f"{len(pp)} postpro folders; using newest: "
          f"{os.path.basename(pp[-1])}  (set TAG to pick another)")
pp_dir = max(pp, key=os.path.getmtime)
run_tag = os.path.basename(pp_dir).replace("postpro_", "", 1)

iters = sorted(
    (d for d in glob.glob(os.path.join(pp_dir, "iteration*"))
     if os.path.isdir(d)),
    key=lambda d: int(re.sub(r"\D", "", os.path.basename(d)) or 0))
if iters:
    POSTPRO = iters[-1]
    print(f"AMR run: {len(iters)} iteration folder(s); using the most "
          f"converged: {os.path.relpath(POSTPRO, RUN_DIR)}")
else:
    POSTPRO = pp_dir
if not os.path.isfile(os.path.join(POSTPRO, "eig.csv")):
    raise FileNotFoundError(f"{POSTPRO} has no eig.csv -- incomplete "
                            f"copy from vanda?")

# ---- palace config (the resolved copy in postpro wins) --------------
cands = (sorted(glob.glob(os.path.join(pp_dir, "*.json")))
         + sorted(glob.glob(os.path.join(POSTPRO, "*.json")))
         + sorted(glob.glob(os.path.join(RUN_DIR,
                                         f"palace_config_{run_tag}.json")))
         + sorted(glob.glob(os.path.join(RUN_DIR, "palace_config*.json"))))
PALACE_CONFIG = None
for c in cands:
    try:
        with open(c, encoding="utf-8") as fh:
            cfg = json.load(fh)
        ports = (cfg.get("Boundaries", {}) or {}).get("LumpedPort", [])
        if any(float(p.get("L") or 0) > 0 for p in ports):
            PALACE_CONFIG = c
            break
    except Exception:
        continue
if PALACE_CONFIG is None:
    raise FileNotFoundError(
        "no palace config with an inductive LumpedPort found -- the EPR "
        "analysis needs the config the solve ran with")
print(f"config: {os.path.relpath(PALACE_CONFIG, RUN_DIR)}")

# ---- junction L / C from the config: what the solver ACTUALLY used --
inductive = [p for p in json.load(open(PALACE_CONFIG, encoding="utf-8"))
             ["Boundaries"]["LumpedPort"]
             if float(p.get("L") or 0) > 0]
LJ = [float(p["L"]) for p in inductive]           # Henries
CJ = [float(p.get("C") or 0.0) for p in inductive] # Farads
for p, l, c in zip(inductive, LJ, CJ):
    print(f"junction port Index {p.get('Index')}: "
          f"L = {l*1e9:g} nH, C = {c*1e15:g} fF, "
          f"attr {p.get('Elements',[{}])[0].get('Attributes')}")

In [ ]:
# ===================== load + validate (pyEPR from here) =============
from IPython.display import display
import importlib
import palace_epr
importlib.reload(palace_epr)   # pick up edits without kernel restart
from palace_epr import (
    load_palace, validate_junction_ports, identify_modes, analyze,
    print_fancy_results, get_g,
)

eig, Pmj = load_palace(POSTPRO)
jj_ports = validate_junction_ports(POSTPRO, LJ, CJ, PALACE_CONFIG)

print(f"\n{len(eig)} modes loaded from "
      f"{os.path.relpath(POSTPRO, RUN_DIR)}\n")
display(eig.style.set_caption("eig.csv").format(
    {"f_GHz": "{:.6f}", "fim_GHz": "{:.4e}", "Q": "{:.4e}"}))
display(pd.DataFrame(Pmj,
                     columns=[f"p[{j+1}]" for j in range(Pmj.shape[1])])
        .style.set_caption("port-EPR.csv  (signed junction "
                           "participations)")
        .format("{:.6e}"))

## Identify the modes, then choose `MODES`

The qubit is the mode with |p| ≈ 1. If |p| is split across two nearly
degenerate modes they are hybridised — analyze the pair together and
compare sums, not per-mode values. The table also shows the Hilbert-space
cost of including each mode count.

In [ ]:
tbl = identify_modes(eig, Pmj, fock_trunc=FOCK_TRUNC)

if MODES is None:
    n = len(eig)
    if n <= 4:
        MODES = list(range(n))
    else:
        qubit = int(np.abs(Pmj).sum(axis=1).argmax())
        MODES = sorted(set([qubit] + [m for m in range(n)
                                      if m != qubit][:3]))
        print(f"\nNOTE: {n} modes found; auto-selected MODES={MODES} "
              f"(qubit mode {qubit} + lowest others). EDIT MODES in the "
              f"first cell if these are not the modes you care about, "
              f"then re-run from there.")
print(f"\nanalyzing MODES = {MODES}")

## Run pyEPR

`analyze` writes the pickle `QuantumAnalysis.__init__` expects, loads it,
and calls the real `analyze_variation`. Participation renormalisation is
off (`renorm_pj=False`): it needs Ansys field integrals Palace does not
produce, and Palace's participations are already normalised.

In [ ]:
res, epra = analyze(
    eig, Pmj,
    Ljs=LJ, Cjs=CJ,
    modes=MODES,
    cos_trunc=COS_TRUNC, fock_trunc=FOCK_TRUNC,
    mode_names=MODE_NAMES,
)

names = MODE_NAMES or [f"Mode {i}" for i in sorted(MODES)]
qubit_ind = print_fancy_results(res, names=names, qubit_ind=None,
                                display_fn=display)
print(f"\nqubit mode: {names[qubit_ind]} (position {qubit_ind} "
      f"within MODES)")

## Perturbative vs exact anharmonicity

`chi_O1` is first-order perturbation theory, `chi_ND` exact
diagonalisation. A large gap on specific modes means PT is breaking down
there — independent evidence of hybridisation. Compare like with like
when checking against HFSS (`compare_with_ansys` in `palace_epr.py` does
this properly, with the sum rules).

In [ ]:
o1 = np.diag(res["chi_O1"].to_numpy())
nd = np.diag(res["chi_ND"].to_numpy())
df = pd.DataFrame({"chi_O1 (MHz)": o1, "chi_ND (MHz)": nd}, index=names)
df["|O1 - ND| (MHz)"] = np.abs(o1 - nd)
df["rel. gap (%)"] = 100 * np.abs(o1 - nd) / np.abs(nd)
display(df.style.set_caption(
    "Perturbative vs exact anharmonicity -- large gap = PT breaking "
    "down").format("{:.4f}"))

## Save the results into this run folder

`epr_results_<tag>.csv` (per-mode), `epr_chi_O1_<tag>.csv` and
`epr_chi_ND_<tag>.csv` (χ matrices, MHz). They travel with the run, and
the registry cell of the main notebook picks the folder up as usual.

In [ ]:
per_mode = pd.DataFrame({
    "f_linear_GHz": np.asarray(res["f_0"], dtype=float).ravel() / 1e3,
    "f_ND_GHz": np.asarray(res["f_ND"], dtype=float).ravel() / 1e3,
    "Q": np.asarray(res["Qs"], dtype=float).ravel(),
    "anharm_ND_MHz": nd,
}, index=names)
per_mode["L_nH"] = LJ[0] * 1e9
per_mode.to_csv(os.path.join(RUN_DIR, f"epr_results_{run_tag}.csv"))
res["chi_O1"].to_csv(os.path.join(RUN_DIR,
                                  f"epr_chi_O1_{run_tag}.csv"))
res["chi_ND"].to_csv(os.path.join(RUN_DIR,
                                  f"epr_chi_ND_{run_tag}.csv"))
print(f"wrote epr_results_{run_tag}.csv, epr_chi_O1_{run_tag}.csv, "
      f"epr_chi_ND_{run_tag}.csv")
print("\nOrder-1 mode-map runs: mode identification, not final "
      "numbers. Quote from the Order-2 track.")